# Train linear heads outside fse
Train scikit-learn classification and regression heads, then import them into fse.
This small synthetic dataset checks API use and numerical agreement. Its scores do not measure real task quality.
Use raw means (`normalize=False`) for filtering: fse can fold a linear head into each word row.
A head trained on normalized means must retain that normalization and the full input dimension.
Install `requirements.txt` from this folder. All generated files stay in a temporary directory.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import os
import platform
from datetime import date

scratch = TemporaryDirectory(prefix="fse-recipe-")
work = Path(scratch.name)
# Third-party caches also belong to this temporary directory.
for name in ("HF_HOME", "XDG_CACHE_HOME", "NUMBA_CACHE_DIR", "MPLCONFIGDIR"):
    os.environ[name] = str(work / name.lower())
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import fse
print(date.today().isoformat(), platform.platform(), platform.machine())

In [ ]:
# Deliberately tiny, hand-built word rows. These demonstrate APIs, not model quality.
words = ["cat", "dog", "pet", "kitten", "puppy", "car", "bus", "road", "train", "truck"]
vectors = np.array([
    [1, 0, .2], [1, .1, 0], [1, 0, 0], [1, 0, .3], [1, .2, 0],
    [0, 1, .2], [0, 1, .1], [0, 1, 0], [0, 1, .3], [0, 1, .4],
], dtype=np.float32)
encoder = fse.Encoder.from_arrays(vectors, words=words, normalize=False)
texts = ["cat pet", "dog puppy", "car road", "bus truck"]

In [ ]:
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error

examples = [f"{a} {b}" for group in (words[:5], words[5:]) for a in group for b in group]
labels = np.repeat(["pet", "vehicle"], 25)
train, test = train_test_split(np.arange(len(examples)), test_size=.3, random_state=7, stratify=labels)
x = encoder.encode(examples, workers=1)
head = LogisticRegression(random_state=7).fit(x[train], labels[train])
classifier = fse.Classifier.from_sklearn(encoder, head)
predicted = classifier.predict([examples[i] for i in test], workers=1)
np.testing.assert_array_equal(predicted, head.predict(x[test]))
np.testing.assert_allclose(classifier.predict_proba(examples, workers=1), head.predict_proba(x), atol=2e-6)
classifier.save(work / "filter.safetensors", standalone=True)
loaded = fse.Classifier.load(work / "filter.safetensors")
np.testing.assert_array_equal(loaded.predict(texts, workers=1), classifier.predict(texts, workers=1))
print("Synthetic held-out classification accuracy:", accuracy_score(labels[test], predicted))

In [ ]:
# A continuous synthetic target with a separate held-out split.
target = (2 * x[:, 0] - x[:, 1] + .5 * x[:, 2]).astype(np.float64)
regression = Ridge(alpha=.01).fit(x[train], target[train])
regressor = fse.Regressor.from_sklearn(encoder, regression)
actual = regressor.predict([examples[i] for i in test], workers=1)
np.testing.assert_allclose(actual.reshape(-1), regression.predict(x[test]), atol=2e-6)
regressor.save(work / "regressor.safetensors", standalone=True)
print("Synthetic held-out regression MAE:", mean_absolute_error(target[test], actual.reshape(-1)))

In [ ]:
scratch.cleanup()